# Football tournaments as the categories graph (Analysis 5, reopened 2026-11-09)

The report's categories graph. It replaces the election slide from
[`02-election-length.ipynb`](02-election-length.ipynb), whose claim did not hold once the
error bars were honest (section 1 below). Decisions and numbers are logged in
`analysis-log.md`, Analysis 4 and Analysis 5, entries of 2026-11-09.

**Why not Poisson error bars:** daily message counts come in bursts. Outside the
tournaments the daily count varies ~27× more than a Poisson count would, so
`sqrt(count)` error bars are far too narrow. Every check here resamples **whole weeks**
(7-day blocks), so a burst stays together.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from wa_analyzer.data import load_own_chat

own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"], utc=True).dt.tz_localize(None)
daily = own.set_index("timestamp").resample("D").size()  # quiet days count as 0

N_BOOT = 2000
SEED = 42
ORANGE, GREY = "#eb6834", "#b0b0b0"  # report focus colour vs context
D28 = pd.Timedelta(days=28)
print(f"{daily.index.min().date()} – {daily.index.max().date()}, {len(daily)} days")

In [ ]:
def week_blocks(values: np.ndarray, size: int = 7) -> list[np.ndarray]:
    """Cut a daily series into consecutive 7-day blocks (the last one may be shorter)."""
    return [values[i : i + size] for i in range(0, len(values), size)]


def boot_means(values: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Mean per day of N_BOOT resamples that draw whole weeks with replacement."""
    blocks = week_blocks(values)
    return np.array([
        np.concatenate([blocks[j] for j in rng.integers(0, len(blocks), len(blocks))]).mean()
        for _ in range(N_BOOT)
    ])


def pct_change_ci(focus: np.ndarray, base: np.ndarray, rng: np.random.Generator) -> tuple[float, float, float]:
    """% change of focus vs base, with a 95% week-bootstrap range."""
    ratio = boot_means(focus, rng) / boot_means(base, rng)
    lo, hi = np.percentile((ratio - 1) * 100, [2.5, 97.5])
    return (focus.mean() / base.mean() - 1) * 100, lo, hi

## 1. Why the election slide was dropped

Same checks on the five ±30-day election windows. Pooled, the group is +9% (95%
−18% to +39%) and pliable-tiger +38% (−10% to +96%, reproducing todo.md section 5).
Only US 2020 stands out — inside the lockdown — and the last two elections are
*quieter* than normal.

In [ ]:
ELECTIONS = {"US 2020": "2020-11-03", "NL 2021": "2021-03-17", "NL 2023": "2023-11-22",
             "US 2024": "2024-11-05", "NL 2025": "2025-10-29"}
W = pd.Timedelta(days=30)
in_election = pd.Series(False, index=daily.index)
for day in ELECTIONS.values():
    in_election |= (daily.index >= pd.Timestamp(day) - W) & (daily.index <= pd.Timestamp(day) + W)

rng = np.random.default_rng(SEED)
rows = []
for who, msgs in (("group", own), ("pliable-tiger", own[own["author"] == "pliable-tiger"])):
    d = msgs.set_index("timestamp").resample("D").size().reindex(daily.index, fill_value=0)
    rest = d[~in_election.to_numpy()].to_numpy()
    windows = {n: d[pd.Timestamp(x) - W : pd.Timestamp(x) + W].to_numpy() for n, x in ELECTIONS.items()}
    windows["pooled"] = np.concatenate(list(windows.values()))
    for name, v in windows.items():
        rows.append((who, name, *pct_change_ci(v, rest, rng)))
pd.DataFrame(rows, columns=["who", "election", "% vs rest", "95% lo", "95% hi"]).round(0)

## 2. Football: the tournaments

Dates checked by the student on 2026-11-09. **World Cup 2026 was not looked at when the
claim was made** (stage 6 of Analysis 5 wrongly said there was no 4th tournament in
the data), so it is a real hold-out.

In [ ]:
TOURNAMENTS = [
    ("Euro 2020\n(summer 2021)", "2021-06-11", "2021-07-11"),
    ("World Cup 2022\n(Nov–Dec)", "2022-11-20", "2022-12-18"),
    ("Euro 2024\n(summer)", "2024-06-14", "2024-07-14"),
    ("World Cup 2026\n(summer, not seen before)", "2026-06-11", "2026-07-19"),
]
in_tournament = pd.Series(False, index=daily.index)
for _, start, end in TOURNAMENTS:
    # hype week before kickoff excluded from 'rest' too, as in Analysis 5
    in_tournament |= (daily.index >= pd.Timestamp(start) - pd.Timedelta(days=7)) & (daily.index <= pd.Timestamp(end))
rest = daily[~in_tournament.to_numpy()]
print(f"rest of the chat: {rest.mean():.1f}/day, var/mean = {rest.var() / rest.mean():.1f} (Poisson = 1)")

## 3. Checks against the rest of the chat

(a) Week bootstrap of the % change vs the rest of the chat. (b) Random-window ranking:
the share of all same-length windows not touching a tournament that are at least as
busy (overall, and within ±1 month of the same season). Windows overlap, so (b) is a
ranking, not a p-value.

In [ ]:
rng = np.random.default_rng(SEED)
vals, ok = daily.to_numpy(), ~in_tournament.to_numpy()
rows = []
for name, start, end in TOURNAMENTS:
    during = daily[start:end]
    pct, lo, hi = pct_change_ci(during.to_numpy(), rest.to_numpy(), rng)
    n = len(during)
    means, months = [], []
    for i in range(len(vals) - n + 1):
        if ok[i : i + n].all():
            means.append(vals[i : i + n].mean())
            months.append(daily.index[i + n // 2].month)
    means, months = np.array(means), np.array(months)
    m = during.index[n // 2].month
    season = np.isin(months, [(m - 2) % 12 + 1, m, m % 12 + 1])
    rows.append((name.split("\n")[0], during.mean(), pct, lo, hi,
                 (means >= during.mean()).mean() * 100, (means[season] >= during.mean()).mean() * 100))
pd.DataFrame(rows, columns=["tournament", "per day", "% vs rest", "95% lo", "95% hi",
                            "% windows as busy", "same season"]).round(1)

## 4. The graph: 4 weeks before kickoff, during, 4 weeks after the final

Student's choice (draft 2): each tournament against the 4 weeks around it, so the
chat's slow decline over the years does not decide the comparison. Draft 1 used the
8 weeks before the hype week; there two burst weeks in April 2024 inflated Euro 2024's
"before" (logged).

In [ ]:
rng = np.random.default_rng(SEED)
PHASES = [("4 weeks before", GREY), ("during", ORANGE), ("4 weeks after", GREY)]
bars, changes = [], []
for name, start, end in TOURNAMENTS:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    parts = {
        "4 weeks before": daily[s - D28 : s - pd.Timedelta(days=1)].to_numpy(),
        "during": daily[s:e].to_numpy(),
        "4 weeks after": daily[e + pd.Timedelta(days=1) : e + D28].to_numpy(),
    }
    for phase, v in parts.items():
        lo, hi = np.percentile(boot_means(v, rng), [2.5, 97.5])
        bars.append((name, phase, v.mean(), lo, hi))
    around = np.concatenate([parts["4 weeks before"], parts["4 weeks after"]])
    changes.append((name.split("\n")[0], around.mean(), parts["during"].mean(),
                    *pct_change_ci(parts["during"], around, rng)))
bars = pd.DataFrame(bars, columns=["tournament", "phase", "mean", "lo", "hi"])
pd.DataFrame(changes, columns=["tournament", "around/day", "during/day", "% change", "95% lo", "95% hi"]).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.4))
x = np.arange(len(TOURNAMENTS))
w = 0.27
for k, (phase, color) in enumerate(PHASES):
    d = bars[bars["phase"] == phase]
    pos = x + (k - 1) * w
    ax.bar(pos, d["mean"], w, color=color, edgecolor="white", linewidth=0.5)
    ax.errorbar(pos, d["mean"], yerr=[d["mean"] - d["lo"], d["hi"] - d["mean"]],
                fmt="none", ecolor="#333333", capsize=2.5, lw=0.9)

# direct labels on the first tournament instead of a legend
first = bars[bars["tournament"] == TOURNAMENTS[0][0]].reset_index(drop=True)
for k, (phase, _) in enumerate(PHASES):
    label = "during the\ntournament" if phase == "during" else phase.replace(" weeks", "\nweeks")
    ax.text(x[0] + (k - 1) * w, first.loc[k, "hi"] + 0.8, label, ha="center", va="bottom", fontsize=8.5,
            color=ORANGE if phase == "during" else "#666666",
            fontweight="bold" if phase == "during" else "normal")

ax.axvline(2.5, color="#999999", lw=0.8, ls=":")  # hold-out from here
ax.set_xticks(x, [t[0] for t in TOURNAMENTS])
ax.set_ylabel("messages per day (average)")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.tick_params(axis="x", length=0)
fig.suptitle("Football tournaments brought activity to the group – until 2026",
             fontsize=14, fontweight="bold", x=0.06, ha="left")
ax.set_title("Average messages per day: 4 weeks before kickoff, during the tournament, 4 weeks after the final",
             fontsize=10, style="italic", loc="left", color="#444444")
fig.text(0.06, 0.01,
         "Error bars: 95% range from resampling whole weeks (2,000×), so bursts of messages stay together.\n"
         "World Cup 2026 was not looked at when the claim was made: a check on new data.",
         fontsize=8, style="italic", color="#444444", ha="left", va="bottom")
fig.tight_layout(rect=(0, 0.07, 1, 0.97))
fig.savefig("football-categories-draft-3.png", dpi=200, bbox_inches="tight")

## Next (2026-11-10)

- Fix the wrapped "4 weeks before/after" labels; label the dotted line before 2026.
- Critique pass (`goad_critique_visual`), then the final image in `final_img/`.
- Write the 1-page report section: what it shows / design and text choices / is it real?